In [221]:
import calendar
from datetime import date
from pathlib import Path

import polars as pl
from unidecode import unidecode

BASE_DIR = Path.cwd().parent / 'data_input' / 'nao_consignados'

In [222]:
retorno = pl.read_excel(f'{BASE_DIR}/08/RETORNO TODOS 08.2026.xlsx', sheet_name='RETORNO_08')
base_ativos = pl.read_excel(f'{BASE_DIR}/08/BASE ATIVOS E CANCELADOS 07.2026.xlsx', sheet_name='ativos_cancelados')
coopart = pl.read_excel(f'{BASE_DIR}/08/ANALITICO COPARTICIPAÇÃO SIGRH - 08 (CONSIGNADO).xlsx', sheet_name='cobranca_titular_com_vinculo')
remuneracao = pl.read_excel(f'{BASE_DIR}/08/REMUNERAÇÃO 07.2026.xlsx')
obitos = pl.read_excel(f'{BASE_DIR}/08/Óbito_07_2026.xlsx')


Could not determine dtype for column 15, falling back to string
Could not determine dtype for column 16, falling back to string
Could not determine dtype for column 20, falling back to string
Could not determine dtype for column 25, falling back to string
Could not determine dtype for column 26, falling back to string
Could not determine dtype for column 44, falling back to string
Could not determine dtype for column 45, falling back to string
Could not determine dtype for column 60, falling back to string
Could not determine dtype for column 61, falling back to string
Could not determine dtype for column 63, falling back to string
Could not determine dtype for column 65, falling back to string


In [223]:
def formata_coluna(cols:list) -> list:
    cols_norm = []
    for col in cols:
        cols_norm.append(unidecode(col.lower().replace(' ', '_')).replace('-', ''))

    return cols_norm

retorno.columns = formata_coluna(retorno.columns)
base_ativos.columns = formata_coluna(base_ativos.columns)
remuneracao.columns = formata_coluna(remuneracao.columns)
coopart.columns = formata_coluna(coopart.columns)
obitos.columns = formata_coluna(obitos.columns)



### Retorno folha

In [224]:
retorno = (retorno.with_columns([
            pl.concat_str([pl.col('cpf'), pl.col('rubrica')], separator='_').alias('chave')]))

In [225]:
cobrados_em_dois_orgaos = retorno.group_by('cpf').agg(
    pl.col('chave').n_unique().alias('chave_unique'),
    pl.col('chave').count().alias('qtd')
).with_columns([
    pl.when(pl.col('chave_unique') == pl.col('qtd')).then(pl.lit('igual')).otherwise(pl.lit('verificar')).alias('teste')
]).filter(pl.col('teste') == 'verificar')


### Remuneração 

In [226]:
remuneracao = remuneracao.with_columns([
    ((4 * pl.col('remuneracao_inas')) / 100).round(2).alias('valor_mensalidade'),
    pl.col('cpf').str.replace_all(r'[^\d]', '')
]).with_columns([
    pl.when(pl.col('valor_mensalidade') < 578).then(578)
    .when(pl.col('valor_mensalidade') > 1538).then(1538)
    .otherwise('valor_mensalidade').alias('valor_mensalidade')
]).filter(pl.col('cod_org') != '911')

### Base ativos

In [227]:
base_ativos1 = base_ativos.with_columns([
    pl.col('cpf_titular').str.replace_all(r'[^\d]', ''),
    pl.col('cpf_segurado').str.replace_all(r'[^\d]', '')
]).select(['cpf_titular', 'cpf_segurado', 'tipo_segurado', 'data_adesao','data_de_cancelamento' , 'codigo_da_empresa', 'matricula_titular_8_digitos', 'idade'])

base_ativos1 = base_ativos1.rename({
    'codigo_da_empresa':'orgao', 'matricula_titular_8_digitos':'matricula'
}).filter(pl.col('orgao') != '911')

base_ativos1 = base_ativos1.with_columns([
    pl.when(pl.col('tipo_segurado') == 'TITULAR').then(pl.lit('41134'))
    .when(pl.col('idade') <= 18).then(pl.lit('41135'))
    .when(pl.col('idade') <= 23).then(pl.lit('41171'))
    .when(pl.col('idade') <= 28).then(pl.lit('41175'))
    .when(pl.col('idade') <= 33).then(pl.lit('41191'))
    .when(pl.col('idade') <= 38).then(pl.lit('41192'))
    .when(pl.col('idade') <= 43).then(pl.lit('41193'))
    .when(pl.col('idade') <= 48).then(pl.lit('41221'))
    .when(pl.col('idade') <= 53).then(pl.lit('41222'))
    .when(pl.col('idade') <= 58).then(pl.lit('41223'))
    .otherwise(pl.lit('41224')).alias('rubrica')
]).with_columns([
    pl.when(pl.col('rubrica') == '41134').then(0)
    .when(pl.col('rubrica') == '41135').then(225)
    .when(pl.col('rubrica') == '41171').then(250)
    .when(pl.col('rubrica') == '41175').then(340)
    .when(pl.col('rubrica') == '41191').then(370)
    .when(pl.col('rubrica') == '41192').then(420)
    .when(pl.col('rubrica') == '41193').then(479)
    .when(pl.col('rubrica') == '41221').then(530)
    .when(pl.col('rubrica') == '41222').then(657)
    .when(pl.col('rubrica') == '41223').then(764)
    .otherwise(853).alias('mensalidade')
])

In [228]:
base_ativos1 = base_ativos1.with_columns([
    pl.concat_str([pl.col('cpf_titular'), pl.col('rubrica')], separator='_').alias('chave')
])
base_ativos1 = base_ativos1.unique(subset=['cpf_titular', 'cpf_segurado'])

In [229]:
base_ativos1 = (base_ativos1.join(
    remuneracao[['cpf', 'valor_mensalidade']], left_on='cpf_titular', right_on='cpf', how='left')
    ).with_columns([
        pl.col('valor_mensalidade').fill_null(578)
    ])

base_ativos1 = base_ativos1.with_columns([
    pl.when(pl.col('mensalidade') == 0).then('valor_mensalidade').otherwise('mensalidade').alias('mensalidade')
]).drop('valor_mensalidade')


In [230]:
base_ativos1 = base_ativos1.filter(pl.col('data_de_cancelamento').is_null())

In [231]:
base_ativos1 = base_ativos1.filter(pl.col('orgao') != '040')

### Pro-rata

In [232]:
mes_corrente = date.today().month - 1

In [233]:
mes = date.today().replace(day=1, month=mes_corrente)  # noqa: DTZ011
total_dias = calendar.monthrange(mes.year, mes.month)[1]

pro_rata = base_ativos1.filter(pl.col('data_adesao') >= mes)
pro_rata = pro_rata.with_columns([
    ((pl.col('mensalidade') / total_dias) * ((total_dias + 1) - pl.col('data_adesao').dt.day())).round(2).alias('valor_pro_rata')
])

pro_rata = pro_rata.with_columns([
    (pl.lit('5') + pl.col('rubrica').str.slice(1)).alias('rubrica')
]).with_columns([
    pl.concat_str(['cpf_titular', 'rubrica'], separator='_').alias('chave')
])

### Copart

In [234]:
coopart = coopart.with_columns([
    pl.col('cpf').str.replace_all(r'[^\d]', '')
]).rename({'cobranca_copart_titular':'titular', 'cobranca_copart_dependente':'dependente'})

In [235]:
coopart

cpf,nome_titular,emp_cod,matricula,sd_rem_ant_titular,sd_rem_ant_dependente,sd_remanescente_anterior,copart_mes_titular,copart_mes_dependente,copart_mes_total,copart_titular,copart_dependente,total_coparticipacao,verifica_copart,valor_a_cobrar,sd_remanescente_seguinte,titular,dependente,remuneracao,10%_remuneracao,ck1,ck2,ck3,ck4,processo,motivo
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,str,str
"""00000089150""","""AILTON FERREIRA FRANCO DE JESU…","""086""","""17209153""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,9085.96,908.59,0,0,0,0,null,null
"""00000332194""","""MÁRIO DE CARVALHO MAGALHÃES""","""028""","""01964305""",0.0,3924.93,3924.93,28.61,197.93,226.54,28.61,4122.86,4151.47,0,1811.26,2340.21,12.48,1798.78,18112.6,1811.26,0,0,0,0,null,null
"""00000419125""","""YUSSIF ZUBLIDI VIEIRA""","""067""","""01760785""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,15639.55,1563.95,0,0,0,0,null,null
"""00000737178""","""VANESSA CRISTIANE DOS SANTOS S…","""552""","""14337193""",0.0,0.0,0.0,98.34,0.0,98.34,98.34,0.0,98.34,0,98.34,0.0,98.34,0.0,14747.61,1474.76,0,0,0,0,null,null
"""00000827169""","""TATIANE FONSECA DE OLIVEIRA""","""552""","""16801199""",0.0,0.0,0.0,9.32,0.0,9.32,9.32,0.0,9.32,0,9.32,0.0,9.32,0.0,10573.02,1057.3,0,0,0,0,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""99965275653""","""CARLA DE ARAUJO COSTA""","""652""","""02106868""",0.0,0.0,0.0,487.0,0.0,487.0,487.0,0.0,487.0,0,487.0,0.0,487.0,0.0,14690.06,1469.0,0,0,0,0,null,null
"""99978695168""","""LÍGIA RIBEIRO MOREIRA""","""652""","""0241287X""",0.0,0.0,0.0,0.0,128.91,128.91,0.0,128.91,128.91,0,128.91,0.0,0.0,128.91,8265.24,826.52,0,0,0,0,null,null
"""99996545172""","""NIRVANA ARTAXERXES SANTOS MATO…","""652""","""02430991""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,11751.44,1175.14,0,0,0,0,null,null


In [236]:
rubrica_coopart = coopart.unpivot(
    index=[ 'cpf'],
    on=['dependente', 'titular'],
    variable_name='tipo_beneficiario',
    value_name='valor'
).with_columns([
    pl.when(pl.col('tipo_beneficiario') == 'dependente')
    .then(pl.lit('41144')).otherwise(pl.lit('41136')).alias('rubrica')
]).with_columns([
    pl.concat_str([pl.col('cpf'), pl.col('rubrica')], separator='_').alias('chave')
]).filter(pl.col('valor') > 0)

### Óbitos

In [237]:
cpf_obitos = (obitos
    .filter(pl.col('motivo_do_desligamento') == '207')
    .group_by('cpf').agg())

### rubricas que não foram coparticipadas

In [252]:
coopart_boleto = rubrica_coopart.join(retorno, on='chave', how='anti')
# chaves = coopart_boleto.group_by('cpf').agg(
#     pl.col('chave').alias('chaves_copart')
# )

coopart_boleto = coopart_boleto.pivot(
    index=['cpf'],
    values='valor',
    on='tipo_beneficiario'
)
coopart_boleto = coopart_boleto.with_columns([
    pl.col('dependente').fill_null(0),
    pl.col('titular').fill_null(0)
]).rename({'dependente':'copartipacao_dependente', 'titular':'copartipacao_titular'})

# coopart_boleto = coopart_boleto.join(chaves, on='cpf', how='inner')

coopart_boleto.with_columns([
    (pl.col('copartipacao_dependente') + pl.col('copartipacao_titular')).alias('total')
]).filter(pl.col('total') > 10)

cpf,copartipacao_dependente,copartipacao_titular,total
str,f64,f64,f64
"""00032177798""",503.74,169.98,673.72
"""00072083751""",397.46,1703.32,2100.78
"""00170860191""",239.22,0.0,239.22
"""00180025104""",50.58,367.32,417.9
"""00200549120""",14.16,1179.22,1193.38
…,…,…,…
"""99956969168""",0.0,140.6,140.6
"""99959275191""",0.0,318.56,318.56
"""99960800130""",0.0,44.61,44.61


In [239]:
pro_rata_boleto = pro_rata.join(retorno, on='chave', how='anti')
# chaves = pro_rata_boleto.group_by('cpf_titular').agg(
#     pl.col('chave').alias('chaves_pro_rata')
# ).rename({'cpf_titular':'cpf'})

pro_rata_boleto = pro_rata_boleto.group_by(['cpf_titular', 'tipo_segurado']).agg(
    pl.col('valor_pro_rata').sum()
)

pro_rata_boleto = pro_rata_boleto.pivot(
    index='cpf_titular',
    values='valor_pro_rata',
    on='tipo_segurado'
)

pro_rata_boleto = pro_rata_boleto.with_columns([
    pl.col('TITULAR').fill_null(0),
    pl.col('DEPENDENTE').fill_null(0)
]).rename({'TITULAR':'pro_rata_titular', 'DEPENDENTE':'pro_rata_dependente', 'cpf_titular':'cpf'})
# pro_rata_boleto = pro_rata_boleto.join(chaves, on='cpf', how='inner')

In [240]:
mensalidade_boletos = base_ativos1.join(retorno, on='chave', how='anti')

# chaves = mensalidade_boletos.group_by('cpf_titular').agg(
#     pl.col('chave').alias('chaves_mensalidade')
# ).rename({'cpf_titular':'cpf'})



mensalidade_boletos = mensalidade_boletos.group_by(['cpf_titular', 'tipo_segurado']).agg(
    pl.col('mensalidade').sum()
)

mensalidade_boletos = mensalidade_boletos.pivot(
    index='cpf_titular',
    values='mensalidade',
    on='tipo_segurado'
)

mensalidade_boletos = mensalidade_boletos.with_columns([
    pl.col('TITULAR').fill_null(0),
    pl.col('DEPENDENTE').fill_null(0)
]).rename({'TITULAR':'mensalidade_titular', 'DEPENDENTE':'mensalidade_dependente', 'cpf_titular':'cpf'})
# mensalidade_boletos = mensalidade_boletos.join(chaves, on='cpf', how='inner')

### Relatorio final 

In [242]:
relatorio_final = (mensalidade_boletos
    .join(coopart_boleto, on='cpf', how='full')
    .with_columns([
        pl.col('cpf').fill_null(pl.col('cpf_right'))
    ]).drop('cpf_right')
    ).fill_null(0)

In [243]:
relatorio_final = (relatorio_final
    .join(pro_rata_boleto, on='cpf', how='full')
    .with_columns([
        pl.col('cpf').fill_null(pl.col('cpf_right'))
    ]).drop('cpf_right')
    ).fill_null(0)

In [244]:
relatorio_final = relatorio_final.with_columns([
    (pl.col('mensalidade_titular') + pl.col('mensalidade_dependente') + pl.col('copartipacao_dependente') + 
     pl.col('copartipacao_titular') + pl.col('pro_rata_titular') + pl.col('pro_rata_dependente') 
     ).alias('total_cobranca')
])

In [247]:
relatorio_final = relatorio_final.join(cpf_obitos, on='cpf', how='anti')

In [249]:
relatorio_final = relatorio_final.filter(pl.col('total_cobranca') > 10)

In [250]:
relatorio_final

cpf,mensalidade_titular,mensalidade_dependente,copartipacao_dependente,copartipacao_titular,pro_rata_titular,pro_rata_dependente,total_cobranca
str,f64,f64,f64,f64,f64,f64,f64
"""00032177798""",0.0,0.0,503.74,169.98,0.0,0.0,673.72
"""00072083751""",0.0,0.0,397.46,1703.32,0.0,0.0,2100.78
"""00180025104""",578.0,853.0,50.58,367.32,0.0,0.0,1848.9
"""00278369839""",578.0,657.0,480.54,233.87,0.0,0.0,1949.41
"""00438090144""",578.0,1617.0,42.0,325.5,0.0,0.0,2562.5
…,…,…,…,…,…,…,…
"""10163468168""",0.0,0.0,0.0,0.0,205.1,0.0,205.1
"""28006127115""",0.0,0.0,0.0,0.0,223.74,0.0,223.74
"""28040295220""",0.0,0.0,0.0,0.0,0.0,566.84,566.84


In [ ]:
# relatorio_debug = pl.read_excel(f'{BASE_DIR}/07/Relatório de Cobrança - NÃO CONSIGNADOS 2026.07.xlsx')
# relatorio_debug.columns = formata_coluna(relatorio_debug.columns)

# relatorio_debug = relatorio_debug.select(['cpf_titular', 'mensalidade_titular', 'pro_rata_titular', 'mensalidade_dependente', 'pro_rata_dependente',
#                         'coparticipacao_titular', 'coparticipacao_dependente', 'total_cobranca'])
# relatorio_debug = relatorio_debug.with_columns([
#     pl.col('cpf_titular').str.replace_all(r'[^\d]', '')
# ])

# relatorio_debug = relatorio_debug.with_columns([
#     pl.col('cpf_titular').str.zfill(11)
# ]).rename({'cpf_titular':'cpf'})

In [ ]:
# relatorio_final.write_excel('relatorio_07_debug.xlsx')

In [ ]:
# retorno = retorno.with_columns([
#     # pl.col('valor').str.replace_all(',', '.').cast(pl.Float64)
# ])

In [ ]:
# retorno = retorno.group_by('cpf', 'rubrica').agg(
#     pl.col('valor').sum()
# )

In [ ]:
rubricas_retorno = retorno.pivot(
    index='cpf',
    values='valor',
    on='rubrica'
)

ComputeError: aggregation 'item' expected no or a single value, got 2 values

This error occurred in the following expression:
	col("valor").filter([(col("rubrica")) ==v (41223)]).item(allow_empty=true)


In [ ]:
rub_mensalidade = base_ativos1.group_by('cpf_titular').agg(
    pl.col('rubrica').alias('rubricas_mensalidade')
)

rub_copart = rubrica_coopart.group_by('cpf').agg(
    pl.col('rubrica').alias('rubricas_copart')
)

rub_pro = pro_rata.group_by('cpf_titular').agg(
    pl.col('rubrica').alias('rubricas_pro')
)

In [ ]:
rubs_gerais = (rub_mensalidade
    .join(rub_copart, left_on='cpf_titular', right_on='cpf', how='full')
    .with_columns([
        pl.when(pl.col('cpf_titular').is_null()).then('cpf').otherwise('cpf_titular').alias('cpf_titular')
    ])
    .drop('cpf')
    )

rubs_gerais = rubs_gerais.join(rub_pro, on='cpf_titular', how='full').drop('cpf_titular_right')

In [ ]:
rubs_gerais = rubs_gerais.join(cpf_obitos, left_on='cpf_titular', right_on='cpf', how='anti')

In [ ]:
rubs_retorno = retorno.group_by('cpf').agg(
    pl.col('rubrica').alias('rubricas_retorno')
)

In [ ]:
rubs_gerais = rubs_gerais.join(rubs_retorno, left_on='cpf_titular', right_on='cpf', how='left')

In [ ]:
cpf_boletos = relatorio_final['cpf'].to_list()

In [ ]:
debug =  rubs_gerais.filter(pl.col('cpf_titular').is_in(cpf_boletos))

In [ ]:
debug.write_excel('relatorio_debug.xlsx')